In [0]:
from pyspark.sql import functions as F

path = "dbfs:/databricks-datasets/sms_spam_collection/data-001/smsData.csv"

raw = (spark.read
       .option("header", "false")
       .option("sep", "\t")     
       .csv(path)
       .toDF("label", "text"))

display(raw.limit(10))
display(raw.groupBy("label").count())

label,text
ham,Ok... But they said i've got wisdom teeth hidden inside n mayb need 2 remove.
ham,� thk of wat to eat tonight.
ham,I dunno until when... Lets go learn pilates...
spam,Someonone you know is trying to contact you via our dating service! To find out who it could be call from your mobile or landline 09064015307 BOX334SK38ch
ham,Ok c � then.
spam,URGENT! We are trying to contact U. Todays draw shows that you have won a �800 prize GUARANTEED. Call 09050003091 from land line. Claim C52. Valid12hrs only
spam,Not heard from U4 a while. Call 4 rude chat private line 01223585334 to cum. Wan 2C pics of me gettin shagged then text PIX to 8552. 2End send STOP 8552 SAM xxx
ham,staff.science.nus.edu.sg/~phyhcmk/teaching/pc1323
ham,Thank god they are in bed!
ham,Hey tmr meet at bugis 930 ?


label,count
ham,1002
spam,322



(a) Model Markowa. Zaimplementuj łańcuch Markowa rzędu k:
- dla każdego k-gramu (ciągu k kolejnych znaków lub słów) oszacuj rozkład prawdo- podobieństwa następnego symbolu na podstawie częstości w danych;
- potraktuj tekst jako ciąg cykliczny (po ostatnim symbolu następuje pierwszy), aby uniknąć “martwych końców”.

Zaimplementuj co najmniej dwa modele: rzędu k = 1 oraz k = 2 lub k = 3.

In [0]:
display(raw.limit(5))
print(raw.columns)

label,text
ham,Ok... But they said i've got wisdom teeth hidden inside n mayb need 2 remove.
ham,� thk of wat to eat tonight.
ham,I dunno until when... Lets go learn pilates...
spam,Someonone you know is trying to contact you via our dating service! To find out who it could be call from your mobile or landline 09064015307 BOX334SK38ch
ham,Ok c � then.


['label', 'text']


In [0]:
tokens_df = (raw
    .select(
        F.col("label"),
        F.lower(F.col("text")).alias("text")        # normalizacja
    )
    .withColumn("text", F.regexp_replace("text", r"\s+", " "))  # czyszczenie
    .withColumn("text", F.trim("text"))                        # czyszczenie
    .withColumn("tokens", F.split("text", " "))                # tokenizacja
    .select("label", "tokens")
    .filter(F.size("tokens") > 2)                              # wiadomości dłuzsze niz 2 slowa
)
tokens_df.show()

+-----+--------------------+
|label|              tokens|
+-----+--------------------+
|  ham|[ok..., but, they...|
|  ham|[�, thk, of, wat,...|
|  ham|[i, dunno, until,...|
| spam|[someonone, you, ...|
|  ham|   [ok, c, �, then.]|
| spam|[urgent!, we, are...|
| spam|[not, heard, from...|
|  ham|[thank, god, they...|
|  ham|[hey, tmr, meet, ...|
| spam|[you, are, a, win...|
| spam|[urgent!, your, m...|
| spam|[loan, for, any, ...|
|  ham|[haha..., sounds,...|
| spam|[you, have, won, ...|
|  ham|[sorry, i, din, l...|
|  ham|[thanx, but, my, ...|
| spam|[free, for, 1st, ...|
| spam|[congratulations,...|
|  ham|[me, n, him, so, ...|
| spam|[pdate_now, -, do...|
+-----+--------------------+
only showing top 20 rows


In [0]:
clean_df = (raw
    .select(
        F.col("label"),
        F.lower(F.col("text")).alias("text")
    )
    .withColumn("text", F.regexp_replace("text", r"[^\x00-\x7F]+", " "))
    .withColumn("text", F.regexp_replace("text", r"[^a-z0-9\.\,\!\?\:\;\' ]+", " "))
    .withColumn("text", F.regexp_replace("text", r"\s+", " "))
    .withColumn("text", F.trim("text"))
)

tokens_df = (clean_df
    .withColumn("tokens", F.split("text", " "))
    .select("label", "tokens")
    .filter(F.size("tokens") > 2)
)

display(tokens_df.limit(15))

label,tokens
ham,"List(ok..., but, they, said, i've, got, wisdom, teeth, hidden, inside, n, mayb, need, 2, remove.)"
ham,"List(thk, of, wat, to, eat, tonight.)"
ham,"List(i, dunno, until, when..., lets, go, learn, pilates...)"
spam,"List(someonone, you, know, is, trying, to, contact, you, via, our, dating, service!, to, find, out, who, it, could, be, call, from, your, mobile, or, landline, 09064015307, box334sk38ch)"
ham,"List(ok, c, then.)"
spam,"List(urgent!, we, are, trying, to, contact, u., todays, draw, shows, that, you, have, won, a, 800, prize, guaranteed., call, 09050003091, from, land, line., claim, c52., valid12hrs, only)"
spam,"List(not, heard, from, u4, a, while., call, 4, rude, chat, private, line, 01223585334, to, cum., wan, 2c, pics, of, me, gettin, shagged, then, text, pix, to, 8552., 2end, send, stop, 8552, sam, xxx)"
ham,"List(staff.science.nus.edu.sg, phyhcmk, teaching, pc1323)"
ham,"List(thank, god, they, are, in, bed!)"
ham,"List(hey, tmr, meet, at, bugis, 930, ?)"


In [0]:
clean_df = (raw
    .select(F.col("label"), F.lower(F.col("text")).alias("text"))
    .withColumn("text", F.regexp_replace("text", r"[^\x00-\x7F]+", " "))
    .withColumn("text", F.regexp_replace("text", r"\b\d[\d,\.]*\b", " <NUM> "))
    .withColumn("text", F.regexp_replace("text", r"\.{2,}", " ... "))
    .withColumn("text", F.regexp_replace("text", r"\!{2,}", " ! "))
    .withColumn("text", F.regexp_replace("text", r"\?{2,}", " ? "))
    .withColumn("text", F.regexp_replace("text", r"[^a-z0-9\<\>\.\,\!\?\:\;\' ]+", " "))
    .withColumn("text", F.regexp_replace("text", r"\s+", " "))
    .withColumn("text", F.trim("text"))
    .withColumn("text", F.regexp_replace("text", r"(https?://\S+|www\.\S+)", " <URL> "))
    .withColumn("text", F.regexp_replace("text", r"\b[\w\.-]+\.(com|org|net|edu|gov|co|uk|pl|io|me|info|biz|sg)\b", " <URL> "))
    .withColumn("text", F.regexp_replace("text", r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " <EMAIL> "))
)

tokens_df = (clean_df
    .withColumn("tokens", F.split("text", " "))
    .select("label", "tokens")
    .filter(F.size("tokens") > 2)
)

display(tokens_df.limit(15))

label,tokens
ham,"List(ok, ..., but, they, said, i've, got, wisdom, teeth, hidden, inside, n, mayb, need, <, >, remove.)"
ham,"List(thk, of, wat, to, eat, tonight.)"
ham,"List(i, dunno, until, when, ..., lets, go, learn, pilates, ...)"
spam,"List(someonone, you, know, is, trying, to, contact, you, via, our, dating, service!, to, find, out, who, it, could, be, call, from, your, mobile, or, landline, <, >, box334sk38ch)"
ham,"List(ok, c, then.)"
spam,"List(urgent!, we, are, trying, to, contact, u., todays, draw, shows, that, you, have, won, a, <, >, prize, guaranteed., call, <, >, from, land, line., claim, c52., valid12hrs, only)"
spam,"List(not, heard, from, u4, a, while., call, <, >, rude, chat, private, line, <, >, to, cum., wan, 2c, pics, of, me, gettin, shagged, then, text, pix, to, <, >, ., 2end, send, stop, <, >, sam, xxx)"
ham,"List(, , , phyhcmk, teaching, pc1323)"
ham,"List(thank, god, they, are, in, bed!)"
ham,"List(hey, tmr, meet, at, bugis, <, >, ?)"


In [0]:
# Model Markowa k=1

from pyspark.sql import Window

def train_markov_k1(tokens_df):
    df = tokens_df.withColumn("n", F.size("tokens"))

    # cykliczność: doklejamy pierwszy token na koniec
    df = df.withColumn("tokens_cyc", F.concat(F.col("tokens"), F.slice(F.col("tokens"), 1, 1)))

    # indeksy 0..n-1
    df = df.withColumn("idx", F.sequence(F.lit(0), F.col("n") - 1))
    df = df.select(F.explode("idx").alias("i"), F.col("tokens_cyc"))

    df = df.withColumn("state", F.element_at("tokens_cyc", F.col("i") + 1))
    df = df.withColumn("next_tok", F.element_at("tokens_cyc", F.col("i") + 2))

    counts = df.groupBy("state", "next_tok").count()

    # prawdopodobieństwa w każdym stanie
    w = Window.partitionBy("state")
    model = (counts
             .withColumn("total", F.sum("count").over(w))
             .withColumn("p", F.col("count") / F.col("total"))
             .select("state", "next_tok", "count", "p"))

    return model

model_k1 = train_markov_k1(tokens_df)

print("Liczba przejść (state,next):", model_k1.count())
display(model_k1.orderBy(F.desc("count")).limit(20))


Liczba przejść (state,next): 13423


state,next_tok,count,p
<,>,1371,1.0
call,<,107,0.49537037037037035
to,<,105,0.22388059701492538
>,.,83,0.060539752005835154
...,i,76,0.09464508094645081
>,<,76,0.055433989788475566
lor,...,63,0.7325581395348837
,,57,0.5
,,57,0.9661016949152542
a,<,56,0.175


In [0]:
# poprawa czyszczenia danych
clean_df2 = (raw
    .select(F.col("label"), F.lower(F.col("text")).alias("text"))
    .withColumn("text", F.regexp_replace("text", r"[^\x00-\x7F]+", " "))
    .withColumn("text", F.regexp_replace("text", r"(https?://\S+|www\.\S+)", " "))
    .withColumn("text", F.regexp_replace("text", r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " "))
    .withColumn("text", F.regexp_replace("text", r"\b[\w\.-]+\.(com|org|net|edu|gov|co|uk|pl|io|me|info|biz|sg)\b", " "))
    .withColumn("text", F.regexp_replace("text", r"\.{2,}", " ... "))
    .withColumn("text", F.regexp_replace("text", r"\!{2,}", " ! "))
    .withColumn("text", F.regexp_replace("text", r"\?{2,}", " ? "))
    .withColumn("text", F.regexp_replace("text", r"[^a-z0-9\.\,\!\?\:\;\' ]+", " "))
    .withColumn("text", F.regexp_replace("text", r"\s+", " "))
    .withColumn("text", F.trim("text"))
    .withColumn("text", F.regexp_replace("text", r"\.\.\.", " "))
)

tokens_df2 = (clean_df2
    .withColumn("tokens_raw", F.split(F.col("text"), " "))
    # usuwamy nulle
    .withColumn("tokens", F.expr("filter(tokens_raw, x -> x is not null and x <> '')"))
    .select("label", "tokens")
    .filter(F.size("tokens") > 2)
)

display(tokens_df2.limit(20))

label,tokens
ham,"List(ok, but, they, said, i've, got, wisdom, teeth, hidden, inside, n, mayb, need, 2, remove.)"
ham,"List(thk, of, wat, to, eat, tonight.)"
ham,"List(i, dunno, until, when, lets, go, learn, pilates)"
spam,"List(someonone, you, know, is, trying, to, contact, you, via, our, dating, service!, to, find, out, who, it, could, be, call, from, your, mobile, or, landline, 09064015307, box334sk38ch)"
ham,"List(ok, c, then.)"
spam,"List(urgent!, we, are, trying, to, contact, u., todays, draw, shows, that, you, have, won, a, 800, prize, guaranteed., call, 09050003091, from, land, line., claim, c52., valid12hrs, only)"
spam,"List(not, heard, from, u4, a, while., call, 4, rude, chat, private, line, 01223585334, to, cum., wan, 2c, pics, of, me, gettin, shagged, then, text, pix, to, 8552., 2end, send, stop, 8552, sam, xxx)"
ham,"List(phyhcmk, teaching, pc1323)"
ham,"List(thank, god, they, are, in, bed!)"
ham,"List(hey, tmr, meet, at, bugis, 930, ?)"


In [0]:
model_k1 = train_markov_k1(tokens_df2)
display(model_k1.orderBy(F.desc("count")).limit(20))

state,next_tok,count,p
you,have,42,0.2441860465116279
this,is,36,0.6
have,won,35,0.3333333333333333
but,i,30,0.26785714285714285
you,are,27,0.1569767441860465
won,a,27,0.6136363636363636
to,contact,27,0.057569296375266525
wan,2,26,0.48148148148148145
i,thk,24,0.05357142857142857
your,mobile,23,0.19491525423728814


In [0]:
# Model Markowa k=2
def train_markov_k2(tokens_df):
    df = tokens_df.withColumn("n", F.size("tokens"))

    df = df.withColumn("tokens_cyc", F.concat(F.col("tokens"), F.slice(F.col("tokens"), 1, 2)))

    df = df.withColumn("idx", F.sequence(F.lit(0), F.col("n") - 1))
    df = df.select(F.explode("idx").alias("i"), F.col("tokens_cyc"))

    df = df.withColumn("w1", F.element_at("tokens_cyc", F.col("i") + 1))
    df = df.withColumn("w2", F.element_at("tokens_cyc", F.col("i") + 2))
    df = df.withColumn("next_tok", F.element_at("tokens_cyc", F.col("i") + 3))

    # łączymy parę w jeden klucz
    sep = " "
    df = df.withColumn("state", F.concat_ws(sep, "w1", "w2"))

    counts = df.groupBy("state", "next_tok").count()

    w = Window.partitionBy("state")
    model = (counts
             .withColumn("total", F.sum("count").over(w))
             .withColumn("p", F.col("count") / F.col("total"))
             .select("state", "next_tok", "count", "p"))

    return model


In [0]:
model_k2 = train_markov_k2(tokens_df2)
display(model_k2.orderBy(F.desc("count")).limit(20))

state,next_tok,count,p
you have,won,22,0.5238095238095238
have won,a,21,0.6
u wan,2,16,0.6956521739130435
this is,the,15,0.4166666666666667
is the,2nd,14,0.875
2nd attempt,to,12,1.0
to receive,a,11,0.6470588235294118
trying to,contact,11,1.0
prize guaranteed.,call,11,1.0
suite342 2lands,row,11,1.0


In [0]:
def train_markov_k3(tokens_df):
    df = tokens_df.withColumn("n", F.size("tokens"))

    df = df.withColumn("tokens_cyc", F.concat(F.col("tokens"), F.slice(F.col("tokens"), 1, 3)))

    df = df.withColumn("idx", F.sequence(F.lit(0), F.col("n") - 1))
    df = df.select(F.explode("idx").alias("i"), F.col("tokens_cyc"))

    df = df.withColumn("w1", F.element_at("tokens_cyc", F.col("i") + 1))
    df = df.withColumn("w2", F.element_at("tokens_cyc", F.col("i") + 2))
    df = df.withColumn("w3", F.element_at("tokens_cyc", F.col("i") + 3))
    df = df.withColumn("next_tok", F.element_at("tokens_cyc", F.col("i") + 4))

    df = df.withColumn("state", F.concat_ws(" ", "w1", "w2", "w3"))

    counts = df.groupBy("state", "next_tok").count()

    w = Window.partitionBy("state")
    model = (counts
             .withColumn("total", F.sum("count").over(w))
             .withColumn("p", F.col("count") / F.col("total"))
             .select("state", "next_tok", "count", "p"))

    return model

In [0]:
model_k3 = train_markov_k3(tokens_df2)
display(model_k3.orderBy(F.desc("count")).limit(20))

state,next_tok,count,p
you have won,a,20,0.9090909090909091
this is the,2nd,14,0.9333333333333333
we are trying,to,10,1.0
are trying to,contact,10,1.0
draw shows that,you,9,1.0
u have won,the,9,0.9
is the 2nd,attempt,9,0.6428571428571429
the 2nd attempt,to,9,1.0
from land line.,claim,8,1.0
shows that you,have,8,0.8888888888888888


(b) Generowanie tekstu.
- Wybierz stan początkowy (np. pierwszy k-gram w tekście źródłowym albo losowy k-gram).
- Wygeneruj dla każdego z wybranych rzędów k tekst o długości co najmniej 500 sym- boli (znaków lub słów).

Do raportu wklej fragmenty (ok. 5–8 wierszy) wygenerowanego tekstu dla różnych wartości k.


In [0]:
from collections import defaultdict
import random

# Konwersja modelu k=1 do słownika

def model_k1_to_dict(model_df):
    rows = model_df.select("state", "next_tok", "p").collect()
    trans = defaultdict(list)
    for r in rows:
        trans[r["state"]].append((r["next_tok"], float(r["p"])))
    return trans

trans_k1 = model_k1_to_dict(model_k1)
print("Liczba stanów (k=1):", len(trans_k1))

# Funkcja losowania next słowa wg prawdopodobieństw

def sample_next(options):
    # options: list[(token, p)] gdzie sum(p)=1
    r = random.random()
    s = 0.0
    for tok, p in options:
        s += p
        if r <= s:
            return tok
    return options[-1][0]  # awaryjnie

# Generator tekstu (k=1)

def generate_k1(trans, start_state=None, length=550):
    if start_state is None:
        start_state = random.choice(list(trans.keys()))

    out = [start_state]
    state = start_state

    for _ in range(length - 1):
        options = trans.get(state)
        if not options:
            # jeśli trafi się stan bez przejść (mało prawdopodobne przy cykliczności), bierzemy losowy
            state = random.choice(list(trans.keys()))
            options = trans[state]

        nxt = sample_next(options)
        out.append(nxt)
        state = nxt

    return " ".join(out)

gen_k1 = generate_k1(trans_k1, length=550)

# 5–8 "wierszy" po ~120 znaków
import textwrap
for line in textwrap.wrap(gen_k1, width=120)[:8]:
    print(line)



Liczba stanów (k=1): 4329
num. dis is your mobile customer. t cs po box 1327 croydon cr9 5wb 0870 is yijue. i will go ah wat to your lesson so
going 2 make contact you! our members up can send hmv1 to 2ham 000 pounds of studentfinancial crisis!spk 2 pick ur gf?
dun c me buy wat? ok lor. ard rest of your mobile vary shop to contact u! call 09050001295 from stereophonics, marley,
dizzee racal, libertines and all time more go elsewhere? yar lor no need. i'll jus anything lor what you had a lot 4
shanghai on orange tariffs and the 2nd attempt to 83222. 18 you are trying to 8007 zed 08701417012 profit 2 study : gd
lor anything lor like dat i thought id txt music to 80082 3 den msg u like dis eve? now closingdate04 09 02 06 03! this
fone love sex. text back 2 is fran i ate spageddies lor i go for details 18 sender: hol just call me bored nope i'm
leaving earlier lor. i jus tot u found dis eve? now 4 tenerife holiday flights inc speak to 86688 now i had a 400


In [0]:
import textwrap

# Konwersja modelu k=2 do słownika
def model_to_dict(model_df):
    rows = model_df.select("state", "next_tok", "p").collect()
    trans = defaultdict(list)
    for r in rows:
        trans[r["state"]].append((r["next_tok"], float(r["p"])))
    return trans

# Generator tekstu (k=2)
def generate_k2(trans, start_state=None, length=550):
    if start_state is None:
        start_state = random.choice(list(trans.keys()))

    w1, w2 = start_state.split(" ", 1)
    out = [w1, w2]
    state = start_state

    for _ in range(length - 2):
        options = trans.get(state)
        if not options:
            state = random.choice(list(trans.keys()))
            w1, w2 = state.split(" ", 1)
            out.extend([w1, w2])
            options = trans[state]

        nxt = sample_next(options)
        out.append(nxt)
        state = out[-2] + " " + out[-1]

    return " ".join(out)

trans_k2 = model_to_dict(model_k2)
gen_k2 = generate_k2(trans_k2, length=550)

for line in textwrap.wrap(gen_k2, width=120)[:8]:
    print(line)


get the official england poly ringtone or colour flag on yer mobile for tonights game! text tone or flag to 84199 now!
opt out to buy tell us then no need to ke qi too bored so i wait 4 me there my lect finishes in an hour anyway. i wanted
to ask to buy mum's present ar. i'm going 4 e lunch? is xy in ur area 4 just 25p free 2 receive 1000 cash or 200 prize.
just call 087104711148 now! only 10p per minute. bt national rate. congratulations! thanks to a party with my girl
friend what r u doing? wat r u doing there? wat u doing there? wat u doing now? i was snoring.they were drunk! u bak at
college if u dun drive then how i come home tonight. it's ok lar. u sleep early too. nite really dun bluff me leh u
sleep early too. nite really dun bluff me leh u sleep early too. nite really dun bluff me leh u sleep early too. nite
really dun bluff me leh u sleep early too nite it's ok yup. anything lor, if u can win 100 of music gift vouchers free


In [0]:
# Konwersja modelu k=3 do słownika
def model_to_dict(model_df):
    rows = model_df.select("state", "next_tok", "p").collect()
    trans = defaultdict(list)
    for r in rows:
        trans[r["state"]].append((r["next_tok"], float(r["p"])))
    return trans

# Generator tekstu (k=2)
def generate_k3(trans, start_state=None, length=550):
    # start_state: string "w1 w2 w3"
    if start_state is None:
        start_state = random.choice(list(trans.keys()))

    w1, w2, w3 = start_state.split(" ", 2)
    out = [w1, w2, w3]
    state = start_state

    for _ in range(length - 3):
        options = trans.get(state)
        if not options:
            # awaryjnie: losowy stan
            state = random.choice(list(trans.keys()))
            w1, w2, w3 = state.split(" ", 2)
            out.extend([w1, w2, w3])
            options = trans[state]

        nxt = sample_next(options)
        out.append(nxt)

        # przesuwamy okno 3-gramu
        state = out[-3] + " " + out[-2] + " " + out[-1]

    return " ".join(out)

trans_k3 = model_to_dict(model_k3)
gen_k3 = generate_k3(trans_k3, length=550)

# 5–8 wierszy do raportu
for line in textwrap.wrap(gen_k3, width=120)[:8]:
    print(line)


dinner going back 2 hotel now. time flies, we're tog 4 exactly a mth today. hope we'll haf many more mths to come thanx.
yup we coming back on sun. finish dinner going back 2 hotel now. time flies, we're tog 4 exactly a mth today. hope we'll
haf many more mths to come thanx. yup we coming back on sun. finish dinner going back 2 hotel now. time flies, we're tog
4 exactly a mth today. hope we'll haf many more mths to come thanx. yup we coming back on sun. finish dinner going back
2 hotel now. time flies, we're tog 4 exactly a mth today. hope we'll haf many more mths to come thanx. yup we coming
back on sun. finish dinner going back 2 hotel now. time flies, we're tog 4 exactly a mth today. hope we'll haf many more
mths to come thanx. yup we coming back on sun. finish dinner going back 2 hotel now. time flies, we're tog 4 exactly a
mth today. hope we'll haf many more mths to come thanx. yup we coming back on sun. finish dinner going back 2 hotel now.



(c) Prosta ocena statystyczna. Zaproponuj co najmniej dwie ilościowe miary jakości wy- generowanego tekstu i policz je:
- porównanie rozkładów n-gramów: np. całkowita wariacja lub odległość KL między rozkładami częstości bigramów/trigramów w tekście źródłowym i generowanym;
- średnia długość słów, udział znaków interpunkcyjnych, udział słów występujących tylko raz (tzw. hapax legomena);
- dowolna inna miara uzasadniona merytorycznie.

In [0]:
from pyspark.sql import functions as F

# źródło jako jeden tekst
src_tokens = tokens_df2.select(F.explode("tokens").alias("tok"))
src_text_df = src_tokens.agg(F.concat_ws(" ", F.collect_list("tok")).alias("text"))

# teksty generowane jako DF (k jako STRING)
gen_df = spark.createDataFrame(
    [("1", gen_k1), ("2", gen_k2), ("3", gen_k3)],
    ["k", "text"]
)

# doklejamy source jako k = 'source'
texts_df = gen_df.unionByName(
    src_text_df.select(F.lit("source").alias("k"), F.col("text"))
)

display(texts_df)

k text source ok but they said i've got wisdom teeth hidden inside n mayb need 2 remove. thk of wat to eat tonight. i dunno until when lets go learn pilates someonone you know is trying to contact you via our dating service! to find out who it could be call from your mobile or landline 09064015307 box334sk38ch ok c then. urgent! we are trying to contact u. todays draw shows that you have won a 800 prize guaranteed. call 09050003091 from land line. claim c52. valid12hrs only not heard from u4 a while. call 4 rude chat private line 01223585334 to cum. wan 2c pics of me gettin shagged then text pix to 8552. 2end send stop 8552 sam xxx phyhcmk teaching pc1323 thank god they are in bed! hey tmr meet at bugis 930 ? you are a winner you have been specially selected to receive 1000 cash or a 2000 award. speak to a live operator to claim call 087123002209am 7pm. cost 10p urgent! your mobile no. was awarded 2000 bonus caller prize on 5 9 03 this is our final try to contact u! call from landline 09064019788 box42wr29c, 150ppm loan for any purpose 500 75,000. homeowners tenants welcome. have you been previously refused? we can still help. call free 0800 1956669 or text back 'help' haha sounds crazy, dunno can tahan anot you have won ?spam 000 cash or a ?2,000 prize! to claim, call09050000327 sorry i din lock my keypad. thanx but my birthday is over already. free for 1st week! no1 nokia tone 4 ur mobile every week just txt nokia to 8077 get txting and tell ur mates. pobox 36504 w45wq 16 norm150p tone congratulations thanks to a good friend u have won the 2,000 xmas prize. 2 claim is easy, just call 08712103738 now! only 10p per minute. bt national rate me n him so funny pdate now double mins and 1000 txts on orange tariffs. latest motorola, sonyericsson nokia bluetooth free! call mobileupd8 on 08000839402 or call2optout !yhl yup no more already thanx 4 printing n handing it up. anything lor. juz both of us lor. it's only 140 ard rest all ard 180 at least which is price 4 2 bedrm 900 oh oh den muz change plan liao go back have to yan jiu again ok lor then we go tog lor okay lor wah like that def they wont let us go haha what did they say in the terms and conditions? dunno lei i thk mum lazy to go out i neva ask her yet that s alrite girl, u know gail is neva wrong ! take care sweet and don t worry.c u l8tr hun!love yaxxx accordingly. i repeat, just text the word ok on your mobile phone and send nah im goin 2 the wrks with j wot bout u? oh i asked for fun. haha take care. ok finishing soon? you are being contacted by our dating service by someone you know! to find out who it is, call from your mobile or landline 09064017305 pobox75ldns7 ay wana meet on sat? wkg on sat? y wan to go there? c doctor? hmv bonus special 500 pounds of genuine hmv vouchers to be won. just answer 4 easy questions. play now! send hmv to 86688 more info: ok im not sure what time i finish tomorrow but i wanna spend the evening with you cos that would be vewy vewy lubly! love me xxx we tried to contact you re your reply to our offer of a video handset? 750 anytime any networks mins? unlimited text? camcorder? reply or call 08000930705 now ur cash balance is currently 500 pounds to maximize ur cash in now send go to 86688 only 150p meg. cc: 08718720201 hg suite342 2lands row w1j6hl where r e meeting tmr? got but got 2 colours lor. one colour is quite light n e other is darker lor. actually i'm done she's styling my hair now. so gd got free ice cream i oso wan aiyah e rain like quite big leh. if drizzling i can at least run home. alright babe, justthought i d sayhey! how u doin?nearly the endof me wk offdam nevamind!we will have 2hook up sn if uwant m8? lovejen x. can i please come up now imin town.dontmatter if urgoin outl8r,just reallyneed 2docd.please dontplease dontignore mycalls,u no thecd isv.important tome 4 2moro no got new job at bar in airport on satsgettin 4.47per hour but means no lie in! keep in touch wat makes u thk i'll fall down. but actually i thk i'm qui

In [0]:
topN = 20

tok_df = (texts_df
    .withColumn("token", F.explode(F.split(F.col("text"), " ")))
    .filter(F.col("token") != "")
)

word_freq = (tok_df
    .groupBy("k", "token")
    .count()
    .withColumn("rank", F.row_number().over(Window.partitionBy("k").orderBy(F.desc("count"))))
    .filter(F.col("rank") <= topN)
    .orderBy("k", "rank")
)

display(word_freq)

k,token,count,rank
1,i,18,1
1,u,14,2
1,go,9,3
1,to,9,4
1,a,7,5
1,the,6,6
1,lor,6,7
1,so,6,8
1,2,6,9
1,me,6,10


In [0]:
tok_len_df = (tok_df
    .withColumn("len", F.length("token"))
)

m_avg_wordlen = (tok_len_df
    .groupBy("k")
    .agg(
        F.avg("len").alias("avg_word_length")
    )
)

display(m_avg_wordlen)

k,avg_word_length
1,3.749090909090909
2,4.0636363636363635
3,3.998181818181818
source,3.956704315088329


In [0]:
len_dist = (tok_len_df
    .groupBy("k", "len")
    .count()
    .orderBy("k", "len")
)

display(len_dist)

k,len,count
1,1,68
1,2,90
1,3,131
1,4,121
1,5,42
1,6,39
1,7,28
1,8,12
1,9,5
1,10,8


In [0]:
m_punct_tokens = (tok_df
    .withColumn("has_punct", F.col("token").rlike(r"[^\w]"))
    .groupBy("k")
    .agg(F.avg(F.col("has_punct").cast("double")).alias("punctuation_share_tokens"))
)

display(m_punct_tokens)

k,punctuation_share_tokens
1,0.10909090909090909
2,0.16
3,0.22545454545454546
source,0.12182643112269524


In [0]:
# z tekstu robimy array tokenów
arr_df = texts_df.withColumn("tokens", F.expr("filter(split(text,' '), x -> x <> '')"))

# bigramy
bi_df = (arr_df
    .withColumn("n", F.size("tokens"))
    .withColumn("i", F.explode(F.sequence(F.lit(1), F.col("n") - 1)))
    .withColumn("w1", F.element_at("tokens", F.col("i")))
    .withColumn("w2", F.element_at("tokens", F.col("i") + 1))
)

m_punct_bigrams = (bi_df
    .withColumn("has_punct_bigram", (F.col("w1").rlike(r"[^\w]") | F.col("w2").rlike(r"[^\w]")).cast("double"))
    .groupBy("k")
    .agg(F.avg("has_punct_bigram").alias("punctuation_share_bigrams"))
)

display(m_punct_bigrams)


k,punctuation_share_bigrams
1,0.20947176684881602
2,0.28233151183970856
3,0.41894353369763204
source,0.23188685620504898


In [0]:
tri_df = (arr_df
    .withColumn("n", F.size("tokens"))
    .withColumn("i", F.explode(F.sequence(F.lit(1), F.col("n") - 2)))
    .withColumn("w1", F.element_at("tokens", F.col("i")))
    .withColumn("w2", F.element_at("tokens", F.col("i") + 1))
    .withColumn("w3", F.element_at("tokens", F.col("i") + 2))
)

m_punct_trigrams = (tri_df
    .withColumn(
        "has_punct_trigram",
        (F.col("w1").rlike(r"[^\w]") | F.col("w2").rlike(r"[^\w]") | F.col("w3").rlike(r"[^\w]")).cast("double")
    )
    .groupBy("k")
    .agg(F.avg("has_punct_trigram").alias("punctuation_share_trigrams"))
)

display(m_punct_trigrams)


k,punctuation_share_trigrams
1,0.29744525547445255
2,0.3813868613138686
3,0.5474452554744526
source,0.3301312994786638


In [0]:
topN = 20

bigram_freq = (bi_df
    .withColumn("bigram", F.concat_ws(" ", "w1", "w2"))
    .groupBy("k", "bigram")
    .count()
    .withColumn("rank", F.row_number().over(Window.partitionBy("k").orderBy(F.desc("count"))))
    .filter(F.col("rank") <= topN)
    .orderBy("k", "rank")
)

display(bigram_freq)

k,bigram,count,rank
1,your mobile,3,1
1,anything lor,3,2
1,then i,3,3
1,you have,2,4
1,dis eve?,2,5
1,so i,2,6
1,i din,2,7
1,will go,2,8
1,den go,2,9
1,i go,2,10


In [0]:
topN = 20

trigram_freq = (tri_df
    .withColumn("trigram", F.concat_ws(" ", "w1", "w2", "w3"))
    .groupBy("k", "trigram")
    .count()
    .withColumn("rank", F.row_number().over(Window.partitionBy("k").orderBy(F.desc("count"))))
    .filter(F.col("rank") <= topN)
    .orderBy("k", "rank")
)

display(trigram_freq)


k,trigram,count,rank
1,jus tot u,2,1
1,dis eve? now,2,2
1,mobile or 5000,1,3
1,smth lor. he,1,4
1,whereare you have,1,5
1,u so i,1,6
1,u found dis,1,7
1,a 400 reward.,1,8
1,awarded a mistake.u,1,9
1,2ham 000 pounds,1,10


In [0]:
token_counts = tok_df.groupBy("k", "token").count()

m_hapax = (token_counts
    .groupBy("k")
    .agg(
        F.sum((F.col("count") == 1).cast("int")).alias("hapax_count"),
        F.count("*").alias("unique_tokens")  # bo to już jest po groupBy(k, token)
    )
    .withColumn("hapax_ratio", F.col("hapax_count") / F.col("unique_tokens"))
    .select("k", "hapax_count", "unique_tokens", "hapax_ratio")
)

display(m_hapax)

k,hapax_count,unique_tokens,hapax_ratio
1,269,346,0.7774566473988439
2,205,301,0.6810631229235881
3,0,30,0.0
source,2558,4329,0.5908985908985909


In [0]:
final_stats = (m_avg_wordlen
    .join(m_punct_tokens, on="k", how="left")
    .join(m_punct_bigrams, on="k", how="left")
    .join(m_punct_trigrams, on="k", how="left")
    .join(m_hapax, on="k", how="left")
    .orderBy(F.when(F.col("k")=="source", 0).otherwise(1), F.col("k"))
)

display(final_stats)


k,avg_word_length,punctuation_share_tokens,punctuation_share_bigrams,punctuation_share_trigrams,hapax_count,unique_tokens,hapax_ratio
source,3.956704315088329,0.12182643112269524,0.23188685620504898,0.3301312994786638,2558,4329,0.5908985908985909
1,3.749090909090909,0.10909090909090909,0.20947176684881602,0.29744525547445255,269,346,0.7774566473988439
2,4.0636363636363635,0.16,0.28233151183970856,0.3813868613138686,205,301,0.6810631229235881
3,3.998181818181818,0.22545454545454546,0.41894353369763204,0.5474452554744526,0,30,0.0


In [0]:
# rozkład bigramów dla każdego tekstu

# Bigramy
bigram_df = (texts_df
    .withColumn("tokens", F.expr("filter(split(text,' '), x -> x <> '')"))
    .withColumn("n", F.size("tokens"))
    .withColumn("i", F.explode(F.sequence(F.lit(1), F.col("n") - 1)))
    .withColumn("w1", F.element_at("tokens", F.col("i")))
    .withColumn("w2", F.element_at("tokens", F.col("i") + 1))
    .withColumn("bigram", F.concat_ws(" ", "w1", "w2"))
)

bigram_freq = bigram_df.groupBy("k", "bigram").count()

# normalizacja
total_counts = bigram_freq.groupBy("k").agg(F.sum("count").alias("total"))
bigram_prob = (bigram_freq
    .join(total_counts, on="k")
    .withColumn("p", F.col("count") / F.col("total"))
)

display(bigram_prob.limit(10))


k,bigram,count,total,p
1,2 study,1,549,0.0018214936247723133
1,you have,2,549,0.0036429872495446266
1,"racal, libertines",1,549,0.0018214936247723133
1,gonna b,1,549,0.0018214936247723133
1,i thought,1,549,0.0018214936247723133
1,"so lousy,",1,549,0.0018214936247723133
1,going 2,1,549,0.0018214936247723133
1,u remember,1,549,0.0018214936247723133
1,ask me,1,549,0.0018214936247723133
1,cost 1.50,1,549,0.0018214936247723133


In [0]:
src_dist = bigram_prob.filter(F.col("k") == "source").select("bigram", F.col("p").alias("p_src"))

tv_list = []

for label in ["1", "2", "3"]:
    gen_dist = bigram_prob.filter(F.col("k") == label).select("bigram", F.col("p").alias("p_gen"))
    
    joined = (src_dist
        .join(gen_dist, on="bigram", how="full")
        .na.fill(0)
        .withColumn("diff", F.abs(F.col("p_src") - F.col("p_gen")))
    )
    
    tv = joined.agg(F.sum("diff").alias("s")).collect()[0]["s"] * 0.5
    tv_list.append((label, tv))

m_tv = spark.createDataFrame(tv_list, ["k", "TV_bigram"])
display(m_tv)

k,TV_bigram
1,0.9254235651879736
2,0.9213206545348891
3,0.9981657575903526


In [0]:
final_stats2 = (final_stats
    .join(m_tv, on="k", how="left")
    .orderBy(F.when(F.col("k")=="source", 0).otherwise(1), F.col("k"))
)

final_stats_fmt = final_stats2.select(
    "k",
    F.round("avg_word_length", 2).alias("avg_word_length"),
    F.round("punctuation_share_tokens", 2).alias("punctuation_share_tokens"),
    F.round("punctuation_share_bigrams", 2).alias("punctuation_share_bigrams"),
    F.round("punctuation_share_trigrams", 2).alias("punctuation_share_trigrams"),
    F.round("hapax_count", 2).alias("hapax_count"),
    F.round("unique_tokens", 2).alias("unique_tokens"),
    F.round("hapax_ratio", 2).alias("hapax_ratio"),
    F.round("TV_bigram", 2).alias("TV_bigram")
)

display(final_stats_fmt)


k,avg_word_length,punctuation_share_tokens,punctuation_share_bigrams,punctuation_share_trigrams,hapax_count,unique_tokens,hapax_ratio,TV_bigram
source,3.96,0.12,0.23,0.33,2558,4329,0.59,null
1,3.75,0.11,0.21,0.3,269,346,0.78,0.93
2,4.06,0.16,0.28,0.38,205,301,0.68,0.92
3,4.0,0.23,0.42,0.55,0,30,0.0,1.0
